# Anomalies and Functional Dependencies

Your Library works but its design hides some problems. The book table stores
authors and genres as comma-separated text, repeats every publisher's details
on each of its books and the member table stores each member's country next
to their city. In this module you will discover why these choices cause
trouble and you will fix them with normalization. We begin by seeing the
problems then introduce the formal tool for reasoning about them: **functional
dependencies**.

## Your Library as it stands

This is the schema you built in Module 4. Open your stage_6_design.dbm in
pgModeler to see it, and notice the book and member tables in particular.

<img src="../../pgmodeler/exports/stage_6_schema.svg" alt="The current Stage 6 Library schema" width="800">

In [ ]:
from rel_db_learning import (
    get_project_connection, run_sql, reset_my_project_data_full, Relation,
)
from rel_db_learning.widgets.self_assessment import multiple_choice, fd_check

project = get_project_connection()
reset_my_project_data_full()

## Seeing the mess

Look at a few books. Notice authors and genres are comma-separated lists and
the publisher details repeat from row to row.

In [ ]:
run_sql("""
    SELECT book_id, title, authors, publisher_name, publisher_city
    FROM book
    ORDER BY book_id
    LIMIT 10
""", project)

## Problem 1: values that are really lists

The authors column holds several authors in one text value. To find books by
a given author, you are forced into fragile substring matching:

In [ ]:
run_sql("SELECT title, authors FROM book WHERE authors LIKE '%Tolkien%'", project)

This works by luck, not design. You cant easily list all distinct authors,
count books per author or join authors to other data. And substring matching
can miss: a search for one name can match another that contains it. A
column should hold a single atomic value. Fixing this is **First Normal
Form**, the next notebook.

## Problem 2: repeated publisher details

Every book carries its publisher's full address. Count how many books repeat
each publisher:

In [ ]:
run_sql("""
    SELECT publisher_name, publisher_city, COUNT(*) AS books
    FROM book
    GROUP BY publisher_name, publisher_city
    ORDER BY books DESC
""", project)

The same publisher address is stored many times. This redundancy causes three
classic anomalies:

- **Update anomaly:** to change a publisher's address you must update every
  book it published. Miss one row and your data contradicts itself.
- **Insert anomaly:** you cannot record a new publisher until it has a book,
  because publisher details live only inside book rows.
- **Delete anomaly:** delete the last book of a publisher and you lose that
  publisher's details entirely.

For example, see how many rows an address change to 'Gollancz' would touch:

In [ ]:
run_sql("SELECT COUNT(*) AS rows_to_update FROM book WHERE publisher_name = 'Gollancz'", project)

## Problem 3: country depends on city

In the member table the country is determined by the city yet both are
stored on every member. The same city always maps to the same country:

In [ ]:
run_sql("SELECT DISTINCT city, country FROM member ORDER BY country, city", project)

If a city's country were ever recorded inconsistently, which member would be
right? Storing country on each member invites exactly that kind of error.

### Try it yourself: cause an anomaly

You just saw the redundancy in the publisher data. Now cause the problems yourself. The table below is the same messy design, and it remembers your changes until you press Reset.

Try each one:

- Update: change the address of a single Gollancz book, then look at the other Gollancz rows. They still hold the old address, so the database now disagrees with itself.
- Delete: remove books one at a time. Delete the last remaining book of a publisher, and that publisher's address is gone, even though we only meant to remove a book.
- Insert: try to record a new publisher that has no book yet. You cannot, because every row needs a book_id.

These three problems, the update anomaly, the delete anomaly, and the insert anomaly, are exactly what normalization removes.

In [ ]:
from rel_db_learning.widgets import anomaly_explorer

anomaly_explorer()

## Functional dependencies

Some columns in a table are determined by others. If you know a book's
publisher_name, you already know that publisher's address and city because a
publisher has one address. We say publisher_name *determines* the address and
the city. This idea, one set of attributes fixing another, is a **functional
dependency**.

We write it as

    X -> Y

read "X determines Y": any two rows that agree on every attribute in X must
also agree on every attribute in Y. Three examples from your Library:

- `publisher_name -> publisher_address, publisher_city`
- `city -> country`
- `book_id -> title, publisher_name, ...` (the primary key determines the whole row)

The first two are the source of the redundancy: a non-key column determines
other columns, so the same fact is repeated on many rows.

## The Relation tool

To reason about functional dependencies precisely, this library gives you a
tool called `Relation`, which you will use in every notebook of this module.
You give it a table's attributes and the dependencies you observe and it
answers two questions: what set of attributes determines (its **closure**)
and what the **candidate keys** are. Let's first see it on a tiny abstract
relation then apply it to your Library.

**A quick warm-up.** Consider an abstract relation R with three attributes A,
B, C and two dependencies: A determines B, and B determines C. We create it,
add the dependencies, and print it back.

In [ ]:
R = Relation(["A", "B", "C"], name="R")
R.add_fd("A", "B")
R.add_fd("B", "C")
print(R)

What does A determine? Follow the arrows: A gives B, and B in turn gives C. So
the **closure** of {A}, everything reachable from A, is all three attributes:

In [ ]:
R.closure(["A"])

What about B? B gives C, but nothing leads back to A, so the closure of {B} is
just {B, C}:

In [ ]:
R.closure(["B"])

A **candidate key** is a minimal set of attributes whose closure is the whole
relation. Here {A} determines everything and no smaller set does, so {A} is the
candidate key. {B} is not, because it cannot reach A.

In [ ]:
R.candidate_keys()

## Back to your Library

Now model the book table the same way. List its columns (we leave out
authors and genres for now, we will fix in the next notebook), add the dependencies and print to check.

In [ ]:
book_rel = Relation(
    ["book_id", "title", "publisher_name", "publisher_address", "publisher_city"],
    name="book",
)
book_rel.add_fd("book_id", ["title", "publisher_name", "publisher_address", "publisher_city"])
book_rel.add_fd("publisher_name", ["publisher_address", "publisher_city"])
print(book_rel)

The closure of {publisher_name} shows it determines other, non-key columns.
That is the formal signature of the redundancy you saw: a non-key column
determining other columns.

In [ ]:
book_rel.closure(["publisher_name"])

And the only candidate key is {book_id}: the primary key determines every
column, and nothing smaller does.

In [ ]:
book_rel.candidate_keys()

## Now the member table

Model member the same way. Here the telltale dependency is `city -> country`:
the city determines the country even though both are stored on every member.

In [ ]:
member_rel = Relation(
    ["member_id", "full_name", "address", "city", "country"],
    name="member",
)
member_rel.add_fd("member_id", ["full_name", "address", "city", "country"])
member_rel.add_fd("city", "country")

member_rel.closure(["city"])

## Exercises

**Exercise 1.** Using `book_rel`, what is the closure of {book_id}? Fill in
your answer as a set of attribute names.

In [ ]:
my_closure = {"???"}
fd_check(book_rel, "closure", student_answer=my_closure, given=["book_id"])

**Exercise 2.** Using `member_rel`, what are the candidate keys? Fill in your
answer as a list of sets.

In [ ]:
my_keys = [{"???"}]
fd_check(member_rel, "candidate_keys", student_answer=my_keys)

In [ ]:
multiple_choice(
    question="A librarian changes a publisher's address but updates only some of the books from that publisher. What kind of anomaly is this?",
    options=["Insert anomaly", "Update anomaly", "Delete anomaly", "No anomaly"],
    correct_index=1,
    explanation="Because the address is repeated across many book rows, updating only some of them leaves the data inconsistent. This is an update anomaly.",
)

In [ ]:
multiple_choice(
    question="In the book relation, is publisher_name a candidate key?",
    options=["Yes", "No"],
    correct_index=1,
    explanation="publisher_name determines the publisher's address and city, but it does not determine book_id or title, so it cannot identify a book. Only book_id is a candidate key.",
)

## What's next

You have seen the anomalies and learned to describe them with functional
dependencies. In the next notebook you will fix the first problem, the
comma-separated authors and genres, by putting your Library into First Normal
Form.